In [ ]:
pip install --upgrade cdflib cdasws

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.7/79.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.1 MB/s eta 0:00:00


In [ ]:
pip install git+https://github.com/hafarooki/PyMFR

  Cloning https://github.com/hafarooki/PyMFR to /tmp/pip-req-build-f12s1nxx
  Running command git clone --filter=blob:none --quiet https://github.com/hafarooki/PyMFR /tmp/pip-req-build-f12s1nxx
  Resolved https://github.com/hafarooki/PyMFR to commit dd1d1402307c9093c96b8252b1c48b32b32fe299
  Preparing metadata (setup.py) ... done
  Created wheel for pymfr: filename=pymfr-2.1.0-py3-none-any.whl size=21812 sha256=8b3538a4b6506150f204b642d2d5452a2bbd8994417e21550566faa31400a3db
  Stored in directory: /tmp/pip-ephem-wheel-cache-xo_ciae2/wheels/38/f9/d9/f667092e0aa7b1882bbfeee888f3bc4e9d2d1175ba6dd55ff0
Successfully built pymfr


In [ ]:
from cdasws import CdasWs
import pandas as pd
import numba
import numpy as np
import numpy.typing as npt
from pymfr.detect import detect_flux_ropes


@numba.njit
def interpolate_points(new_times: npt.NDArray[np.datetime64],
                original_times: npt.NDArray[np.datetime64],
                original_values: npt.NDArray[np.float64],
                max_separation: np.timedelta64):
    nonan_index = ~np.isnan(original_values)
    original_times = original_times[nonan_index]
    original_values = original_values[nonan_index]

    interpolated = np.zeros(len(new_times)) * np.nan

    i_original = 0

    for i_new in range(len(new_times)):
        time_new = new_times[i_new]

        # leave as nan if only sample after, not before
        if original_times[i_original] > time_new:
            continue

        # bring i_original to right before/at the new time
        while i_original + 1 < len(original_times) and original_times[i_original + 1] <= time_new:
            i_original += 1

        # leave as nan if only sample before, not after
        if i_original + 1 == len(original_times):
            continue

        time_left, time_right = original_times[i_original], original_times[i_original + 1]

        # skip if samples too far apart
        if time_right - time_left > max_separation:
            continue

        value_left, value_right = original_values[i_original], original_values[i_original + 1]

        interpolated[i_new] = value_left + (value_right - value_left) * ((time_new - time_left) / (time_right - time_left))

    return interpolated


def get_data_wind(interval_start, interval_end):
  plasma_properties = ["Proton_VX_nonlin",
                        "Proton_VY_nonlin",
                        "Proton_VZ_nonlin",
                        "Proton_W_nonlin",
                        "Proton_Np_nonlin"]
  response, plasma_data = CdasWs().get_data("WI_H1_SWE",
                                            plasma_properties,
                                            interval_start, interval_end)
  assert response["http"]["status_code"] == 200, f"WIND plasma data unavailable for interval {interval_start}--{interval_end} (response: {response})"
  response, mag_data = CdasWs().get_data("WI_H0_MFI",
                                         ["BGSE"],
                                         interval_start, interval_end)
  assert response["http"]["status_code"] == 200, f"WIND mag data unavailable for interval {interval_start}--{interval_end} (response: {response})"
  time_plasma = plasma_data["Epoch"]
  time_mag = mag_data["Epoch"]
  plasma_interpolated = {x: interpolate_points(time_mag.values,
                                               time_plasma.values,
                                               plasma_data[x].values,
                                               max_separation=np.timedelta64(120, "s"))
                          for x in plasma_properties}
  # (1 km/s ** 2)(proton mass)/(2k_B) = 60.5737564 K
  temperature = 60.5737564 * plasma_interpolated["Proton_W_nonlin"] ** 2
  return pd.DataFrame({"Bx": mag_data["BGSE"][:, 0],
                       "By": mag_data["BGSE"][:, 1],
                       "Bz": mag_data["BGSE"][:, 2],
                       "Tp": temperature,
                       "Np": plasma_interpolated["Proton_Np_nonlin"],
                       "Vpx": plasma_interpolated["Proton_VX_nonlin"],
                       "Vpy": plasma_interpolated["Proton_VY_nonlin"],
                       "Vpz": plasma_interpolated["Proton_VZ_nonlin"]},
                      index=pd.to_datetime(time_mag, utc=True))


def get_data_mms(interval_start, interval_end):
  plasma_properties = ["mms1_dis_bulkv_gse_fast"]
  plasma_properties_v2 = ["mms1_des_numberdensity_fast"]
  plasma_properties_v3 = ["T"] #["T1800"] #["T"]

  response, plasma_data = CdasWs().get_data("MMS1_FPI_FAST_L2_DIS-MOMS",
                                            plasma_properties,
                                            interval_start, interval_end)
  assert response["http"]["status_code"] == 200, f"MMS plasma data unavailable for interval {interval_start}--{interval_end} (response: {response})"

  response, plasma_data_v2 = CdasWs().get_data("MMS1_FPI_FAST_L2_DES-MOMS",
                                            plasma_properties_v2,
                                            interval_start, interval_end)
  assert response["http"]["status_code"] == 200, f"MMS plasma2 data unavailable for interval {interval_start}--{interval_end} (response: {response})"

  response, mag_data = CdasWs().get_data("MMS1_FGM_SRVY_L2",
                                         ["mms1_fgm_b_gse_srvy_l2_clean"],
                                         interval_start, interval_end)
  assert response["http"]["status_code"] == 200, f"MMS mag data unavailable for interval {interval_start}--{interval_end} (response: {response})"

  response, plasma_data_v3 = CdasWs().get_data("OMNI_HRO2_1MIN", #"OMNI2_H0_MRG1HR", #"OMNI_HRO2_1MIN",
                                            plasma_properties_v3,
                                            interval_start, interval_end)
  assert response["http"]["status_code"] == 200, f"OMNI data unavailable for interval {interval_start}--{interval_end} (response: {response})"

  time_plasma = plasma_data["Epoch"]
  time_plasma_v2 = plasma_data_v2["Epoch"]
  time_plasma_v3 = plasma_data_v3["Epoch"]

  time_mag = mag_data["Epoch"]
  df_mag = pd.DataFrame({"Bx": mag_data["mms1_fgm_b_gse_srvy_l2_clean"][:, 0],
                         "By": mag_data["mms1_fgm_b_gse_srvy_l2_clean"][:, 1],
                         "Bz": mag_data["mms1_fgm_b_gse_srvy_l2_clean"][:, 2]},
                      index=pd.to_datetime(time_mag, utc=True))

  df_plasma = pd.DataFrame({"Vpx": plasma_data["mms1_dis_bulkv_gse_fast"][:, 0],
                            "Vpy": plasma_data["mms1_dis_bulkv_gse_fast"][:, 1],
                            "Vpz": plasma_data["mms1_dis_bulkv_gse_fast"][:, 2]},
                      index=pd.to_datetime(time_plasma, utc=True))
  df_plasma_v2 = pd.DataFrame({"Np": plasma_data_v2["mms1_des_numberdensity_fast"]},
                      index=pd.to_datetime(time_plasma_v2, utc=True))
  df_plasma_v3 = pd.DataFrame({"Tp": plasma_data_v3["T"]},
                      index=pd.to_datetime(time_plasma_v3, utc=True))

  df_mag = df_mag.shift(0.5, freq='60s').resample('60s').mean()
  df_plasma = df_plasma.shift(0.5, freq='60s').resample('60s').mean()
  df_plasma = df_plasma.reindex(df_mag.index)
  df_plasma_v2 = df_plasma_v2.shift(0.5, freq='60s').resample('60s').mean()
  df_plasma_v2 = df_plasma_v2.reindex(df_mag.index)
  df_plasma_v3 = df_plasma_v3.shift(0.5, freq='60s').resample('60s').mean()
  df_plasma_v3 = df_plasma_v3.reindex(df_mag.index)

  df_plasma["Np"] = df_plasma_v2["Np"]
  df_plasma["Tp"] = df_plasma_v3["Tp"]

  return pd.concat([df_mag, df_plasma], axis=1)


def get_data(spacecraft, interval_start, interval_end):
  match spacecraft:
    case "WIND": df = get_data_wind(interval_start, interval_end)
    case "MMS": df = get_data_mms(interval_start, interval_end)
  df["filled"] = df.isna().any(axis=1)
  df = df.interpolate().bfill().ffill()
  return df

def spherical(x, y, z):
  altitude = np.rad2deg(np.arctan2(np.sqrt(x ** 2 + y ** 2), z))
  azimuth = np.rad2deg(np.arctan2(y, x))
  return altitude, azimuth

In [ ]:
from matplotlib.patches import Patch
from matplotlib.dates import DateFormatter
from scipy import ndimage
import scipy
from scipy import constants as const
import matplotlib.pyplot as plt
from google.colab import files
import io

In [ ]:
spacecraft = "MMS"
#interval_start = "2019-04-13T15:01:30Z"
#interval_end = "2019-04-13T23:59:30Z"

uploaded = files.upload()

In [ ]:
listf = pd.read_csv(io.BytesIO(uploaded['MMS_examples.csv']))
st_time = pd.DataFrame(listf, columns = ['start'])
et_time = pd.DataFrame(listf, columns = ['end'])

n1 = len(st_time)
interval_start = []
interval_end = []

for i in range(0, n1, 1):
  a1 = str(st_time[i:i+1])
  b1 = str(et_time[i:i+1])

  a2 = a1[26:47]
  b2 = b1[26:47]

  if a2[0] == '2':
    interval_start.append(a2)
  elif a2[0] != '2':
    interval_start.append(a1[28:47])

  if b2[0] == '2':
    interval_end.append(b2)
  elif b2[0] != '2':
    interval_end.append(b1[28:47])


In [ ]:
########## parameters ##########

slist = []
elist = []
duration_list = []
average_B_list = []
axis_MFR_list = []
altitude_list = []
azimuth_list = []
beta_list = []
diameter_list = []
#error_residue_list = []
#error_fit_list = []
#alfvenicity_walen_slope_list = []
#frame_quality_list = []
#flow_field_alignment_list = []

average_V_list = []
average_n_list = []

#interval_start = "2020-04-22T13:24:30Z"
#interval_end = "2020-04-22T23:59:30Z"   # and also n2 / window_length_max =len(df)
n2 = len(interval_start)

for i_time in range(n2):
  df = get_data(spacecraft, interval_start[i_time], interval_end[i_time])
  magnetic_field = df[["Bx", "By", "Bz"]].values
  proton_velocity = df[["Vpx", "Vpy", "Vpz"]].values
  proton_density = df["Np"].values
  proton_gas_pressure = proton_density * df["Tp"] * 1.381e-8  # (1 per cc)(k_B)(1 K) = 1.381e-8 nPa
  window_length_min = 10
  window_length_max = len(df)#720 #len(df)
  window_lengths = list(np.unique((10 ** np.arange(np.log10(window_length_min), np.log10(window_length_max), 0.02)).astype(int)))

  output = detect_flux_ropes(magnetic_field,
                              proton_velocity,
                              proton_density,
                              proton_gas_pressure,
                              sample_spacing=60,
                              batch_size=int(1e5),
                              window_lengths=window_lengths,
                              window_steps=[1 for x in window_lengths],
                              threshold_flow_field_alignment=0.8,
                              n_trial_axes=256,
                              max_processing_resolution=None,
                              threshold_diff=0.3,
                              threshold_fit=0.3,
                              cuda = False)
  if output is None:
    print("No events detected")
  else:
    print(len(output.event), "events detected")

  a = output.window_length.values.shape[0]
  y = 0

  for i in range(a):
    event = output.sel(event = i)
    resolution = event.map_Az.shape[-1]
    window_start = int(event.window_start.item())
    window_length = int(event.window_length.item())
    assert window_start + window_length <= len(df)
    temporal_scale = event.temporal_scale.item()
    residue = event.residue_diff.item()
    error_fit = event.residue_fit.item()

    axis = event[["axis_x", "axis_y", "axis_z"]].to_array().values
    propagation_velocity = event[["velocity_x", "velocity_y", "velocity_z"]].to_array().values

    event_flow_velocity = proton_velocity[window_start:window_start + window_length, :]
    event_magnetic_field = magnetic_field[window_start:window_start + window_length, :]
    event_density = proton_density[window_start:window_start + window_length]
    Pgas = proton_gas_pressure[window_start:window_start + window_length]

    x_unit = -(propagation_velocity - np.dot(propagation_velocity, axis) * axis)
    x_unit = x_unit / np.linalg.norm(x_unit)
    y_unit = np.cross(axis, x_unit)
    rotation_matrix = np.column_stack([x_unit, y_unit, axis])
    rotation_matrix = rotation_matrix.T  # transpose gives inverse of rotation matrix

    rotated_magnetic_field = event_magnetic_field @ rotation_matrix.T

    dx = event.spatial_scale.item() / window_length
    A = scipy.integrate.cumulative_trapezoid(-rotated_magnetic_field[:, 1], initial=0, dx=dx)
    alpha = event.walen_slope.item() ** 2

    width = event.spatial_scale.item()

    A_peak = A[np.abs(A).argmax()]
    A_sign = np.sign(A_peak)

    map_core_mask = event.map_core_mask.values.astype(bool)
    map_A = event.map_Az.values
    map_Bx = event.map_Bx.values
    map_By = event.map_By.values

    area = np.sum(map_core_mask.astype(int)) * (width / map_A.shape[0]) ** 2
    # pi r^2 = A -> r = sqrt(A/pi) -> d = 2sqrt(A/pi)
    diameter = 2 * np.sqrt(area / np.pi)

    altitude, azimuth = spherical(*axis)
    Bmag = np.linalg.norm(event_magnetic_field, axis=1)
    P_B = Bmag ** 2 * 3.97887358e-4

    time = df.index.values[window_start:window_start + window_length]
    time_interp = time[0] + np.linspace(0, 1, resolution) * np.timedelta64(int(temporal_scale), 's')

    st = np.datetime_as_string(time[0])
    et = np.datetime_as_string(time[-1])

    slist.append(st.replace('T', '/')[0:19])
    elist.append(et.replace('T', '/')[0:19])
    duration_list.append(temporal_scale / 60)
    average_B_list.append(f"{np.linalg.norm(event_magnetic_field, axis=1).mean():.3f}")
    axis_MFR_list.append(axis)
    altitude_list.append(f"{altitude:.2f}")
    azimuth_list.append(f"{azimuth:.2f}")
    beta_list.append((Pgas / P_B).mean())
    diameter_list.append(10**(np.log10(diameter)))
    #error_residue_list.append(f"{residue:.3f}")
    #error_fit_list.append(f"{error_fit:.3f}")
    #alfvenicity_walen_slope_list.append(f"{event.walen_slope.item():.3f}")
    #frame_quality_list.append(event.frame_quality.item())
    #flow_field_alignment_list.append(event.flow_field_alignment.item())

    average_V_list.append(f"{np.linalg.norm(event_flow_velocity, axis=1).mean():.3f}")
    average_n_list.append(f"{event_density.mean():.3f}")


result_s = pd.DataFrame(slist)
result_e = pd.DataFrame(elist)
result_duration_list = pd.DataFrame(duration_list)
result_average_B_list = pd.DataFrame(average_B_list)
result_axis_MFR_list =pd.DataFrame(axis_MFR_list)
result_altitude_list = pd.DataFrame(altitude_list)
result_azimuth_list = pd.DataFrame(azimuth_list)
result_beta_list = pd.DataFrame(beta_list)
result_diameter_list = pd.DataFrame(diameter_list)
result_error_residue_list = pd.DataFrame(error_residue_list)
result_error_fit_list = pd.DataFrame(error_fit_list)
result_alfvenicity_walen_slope_list = pd.DataFrame(alfvenicity_walen_slope_list)
result_frame_quality_list = pd.DataFrame(frame_quality_list)
result_flow_field_alignment_list = pd.DataFrame(flow_field_alignment_list)

result_average_V_list = pd.DataFrame(average_V_list)
result_average_n_list = pd.DataFrame(average_n_list)

'''
out1 = np.hstack([result_s, result_e, result_duration_list, result_average_B_list, result_axis_MFR_list, result_altitude_list,
                  result_azimuth_list, result_beta_list, result_diameter_list, result_error_residue_list, result_error_fit_list,
                  result_alfvenicity_walen_slope_list, result_frame_quality_list, result_flow_field_alignment_list, result_average_V_list,
                  result_average_n_list])
'''
out1 = np.hstack([result_s, result_e, result_duration_list, result_average_B_list, result_axis_MFR_list, result_altitude_list,
                  result_azimuth_list, result_beta_list, result_diameter_list, result_average_V_list, result_average_n_list])
pd.DataFrame(out1).to_csv('result_9B_0.csv')
#files.download('result_9B_0.csv')

In [ ]:
  #fig, axs = plt.subplot_mosaic([["BB", "BB"],
  #                                 ["B", "B"],
  #                                 ["map", "map"],
  #                                 ["map", "map"]], figsize=(6, 9.5), layout = "constrained")
  fig, axs = plt.subplot_mosaic([["BB", "BB"],
                                   ["B", "B"],
                                   ["map", "map"]], figsize=(6, 9.5), layout = "constrained", gridspec_kw = {"height_ratios": [1,1,2]})

  axs["BB"].set_box_aspect(.3)
  #axs["map"].set_aspect("equal")
  axs["B"].set_box_aspect(.3)
  # axs["map"].yaxis.tick_right()
  # axs["map"].yaxis.set_label_position("right")
  #fig.subplots_adjust(hspace=0.03)


  colors = ["red", "green", "blue"]

  st = np.datetime_as_string(time[0])
  et = np.datetime_as_string(time[-1])

  response, mm = CdasWs().get_data("MMS1_FGM_SRVY_L2",["mms1_fgm_b_gse_srvy_l2_clean"],st,et)
  assert response["http"]["status_code"] == 200, f"MMS mag data unavailable for interval {interval_start}--{interval_end} (response: {response})"

  time_mag = mm["Epoch"]
  df_mag = pd.DataFrame({"Bx": mm["mms1_fgm_b_gse_srvy_l2_clean"][:, 0],
                         "By": mm["mms1_fgm_b_gse_srvy_l2_clean"][:, 1],
                         "Bz": mm["mms1_fgm_b_gse_srvy_l2_clean"][:, 2]},
                        index=pd.to_datetime(time_mag, utc=True))
  Bt = np.sqrt(df_mag["Bx"] ** 2 + df_mag["By"] ** 2 + df_mag["Bz"] ** 2)
  axs["BB"].plot(time_mag, Bt, color = "black", label=r"$B$", linewidth = 0.3)
  axs["BB"].plot(time_mag, df_mag["Bx"], color = "red", label=r"$B_{x,\mathrm{GSE}}$", linewidth = 0.3)
  axs["BB"].plot(time_mag, df_mag["By"], color = "green", label=r"$B_{y,\mathrm{GSE}}$", linewidth = 0.3)
  axs["BB"].plot(time_mag, df_mag["Bz"], color = "blue", label=r"$B_{z,\mathrm{GSE}}$", linewidth = 0.3)
  axs["BB"].xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
  axs["BB"].xaxis.set_tick_params(rotation=0)
  start_time = pd.Timestamp(time[0])
  axs["BB"].set_ylabel(r"[MMS] $\mathrm{B}$ [nT]")
  axs["BB"].set_xlim(time[0], time[-1])
  #axs["BB"].set_title(f"Time ({st.replace('T', '/')[0:19]} to {et.replace('T', '/')[0:19]})", loc = "center", pad = 20)
  axs["BB"].set_title(f"Time (Date: 1 November 2017)", loc = "center", pad = 20)

  #axs["BB"].legend(loc="lower center", ncol=4, fontsize=10, labelcolor='linecolor', handlelength=0, frameon=False, borderaxespad=0.2, borderpad=0.1).set_in_layout(False)
  axs["BB"].legend(loc="upper left", bbox_to_anchor = (1.0, 0.85), ncol = 1, fontsize=10, labelcolor='linecolor', handlelength=0, frameon=False, borderaxespad=0.05, borderpad=0.05)
  axs["BB"].text(0.015, 0.85, "(a)", color="black", fontsize=15, transform = axs["BB"].transAxes)

  for i_color, color in enumerate(colors):
      axs["B"].plot(time, rotated_magnetic_field[:, i_color], color=color)
  axs["B"].plot(time, np.linalg.norm(rotated_magnetic_field, axis=1), color="black", label=r"$B$")

  axs["B"].plot(time_interp, event.map_Bx[resolution // 2, :], color="red", ls="--", label=r"$B_{x,\mathrm{FR}}$")
  axs["B"].plot(time_interp, event.map_By[resolution // 2, :], color="green", ls="--", label=r"$B_{y,\mathrm{FR}}$")
  axs["B"].plot(time_interp, event.map_Bz[resolution // 2, :], color="blue", ls="--", label=r"$B_{z,\mathrm{FR}}$")
  axs["B"].xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
  axs["B"].xaxis.set_tick_params(rotation=0)
  start_time = pd.Timestamp(time[0])
  #axs["B"].set_xlabel(f"Time ({st.replace('T', '/')[0:19]} to {et.replace('T', '/')[0:19]})")
  #axs["B"].set_xlabel(f"Time (Date: {start_time.day} {start_time.month_name()} {start_time.year})")
  axs["B"].set_xlabel(f" ", labelpad = 20)
  axs["B"].set_ylabel(r"[MMS] $\mathrm{B}$ [nT]")
  axs["B"].set_xlim(time[0], time[-1])

  #axs["B"].legend(loc="lower center", ncol=4, fontsize=10, labelcolor='linecolor', handlelength=0, frameon=False, borderaxespad=0.2, borderpad=0.1).set_in_layout(False)
  axs["B"].legend(loc="upper left", bbox_to_anchor = (1.0, 0.85), ncol = 1, fontsize=10, labelcolor='linecolor', handlelength=0, frameon=False, borderaxespad=0.05, borderpad=0.05)


  Blim = np.linalg.norm(event_magnetic_field, axis=1).max() * 1.1
  axs["B"].set_ylim(-Blim, Blim)
  axs["B"].text(0.015, 0.85, "(b)", color="black", fontsize=15, transform = axs["B"].transAxes)

  extent = (0, 1, -.5, .5)

  closed_min = (map_A[map_core_mask] * A_sign).min()
  closed_max = (map_A[map_core_mask] * A_sign).max()
  image = axs["map"].imshow(np.where(map_A * A_sign > 0, event.map_Bz, 0),
                            vmin=max(0, event.map_Bz.min() * 0),
                            vmax=event.map_Bz.values[map_core_mask].max(),
                            interpolation="bicubic",
                            extent=extent,
                            origin="lower",
                            rasterized=True,
                            cmap="gist_heat",
                            aspect = "auto")
  # colorbar = fig.colorbar(image, location="left", shrink=0.5, ax=axs["map"])
  # colorbar.set_label("$B_z$ (nT)")
  enclosed_levels = (map_A[map_core_mask] * A_sign)
  levels = np.linspace(0, enclosed_levels.max(), 10)
  axs["map"].contour(ndimage.zoom(map_A * A_sign, 100), linewidths=1, origin="lower", extent=extent, levels=levels, colors="black", antialiased=True)
  axs["map"].set_xlim(extent[0], extent[1])
  axs["map"].set_ylim(extent[2], extent[3])
  axs["map"].set_xlabel(f"$x/{{\\Delta x}}$ [$\\Delta x = 10^{{{np.log10(width):.1f}}}$ km]")
  axs["map"].set_ylabel("$y/{{\\Delta x}}$")
  axs["map"].axhline(0, color="aqua", lw=2, ls="--")
  axs["map"].text(.1, .06, "SC path", color="aqua", fontsize=10, bbox=dict(facecolor='black', alpha=0.5, edgecolor='black'))
  axs["map"].set_rasterization_zorder(2)
  axs["map"].text(0.015, 0.94, "(c)", color="white", fontsize=15, transform = axs["map"].transAxes)

  plt.show()

  slist.append(st.replace('T', '/')[0:19])
  elist.append(et.replace('T', '/')[0:19])


  #fig.savefig(str(y) + '_' + str(time[0])[0:4] + str(time[0])[5:7] + str(time[0])[8:10] + '_' + str(time[0])[11:13] + str(time[0])[14:16] + '_' + str(time[-1])[11:13] + str(time[-1])[14:16] + '.png')#.format(y=y))#, bbox_inches='tight')
  #files.download(str(y) + '_' + str(time[0])[0:4] + str(time[0])[5:7] + str(time[0])[8:10] + '_' + str(time[0])[11:13] + str(time[0])[14:16] + '_' + str(time[-1])[11:13] + str(time[-1])[14:16] + '.png')

  #plt.show()
  #y+=1


In [ ]:
    import matplotlib.ticker as ticker

    fig, axs = plt.subplot_mosaic([["A", "A"],
                                   ["PA", "PA"]], figsize=(6, 7), constrained_layout = False)# layout="constrained")
    fig.subplots_adjust(hspace = 0.24)
    dx = np.linalg.norm(np.dot(propagation_velocity, x_unit))
    rotated = rotation_matrix @ event_magnetic_field.T
    A = scipy.integrate.cumulative_trapezoid(-rotated[1], initial=0)

    aa = abs(A/min(A)) #if the plots are wrong, aa = abs(A/max(A))
    ####################################
    #before plotting, it should be done
    #aa = np.array(aa)
    #idx = np.where(aa == 1)
    #idx + 1 --> a01[0:idx + 1]
    #len(aa) --> a02[idx + 1:len(aa)]
    ####################################

    a01 = np.zeros(len(aa))
    a02 = np.zeros(len(aa))
    a01[0:195] = aa[0:195]
    a01[195:363] = np.nan
    a02[0:195] = np.nan
    a02[195:363] = aa[195:363]
    #a01[0:127] = aa[0:127]
    #a01[127:317] = np.nan
    #a02[0:127] = np.nan
    #a02[127:317] = aa[127:317]

    #axs["A"].plot(time, aa, color = 'black', linewidth = 1.0)
    axs["A"].plot(time, a01, '-', color = 'red', label = '1st half path', linewidth = 1.0)
    axs["A"].plot(time, a02, '-', color = 'blue', label = '2nd half path', linewidth = 1.0)
    axs["A"].set_xlabel("Time", fontsize=10)#, weight ='bold')
    axs["A"].set_ylabel(r"$|A/A_{peak}|$", fontsize=10)#, weight ='bold')
    axs["A"].tick_params(axis = 'both', labelsize = 10)
    axs["A"].xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
    axs["A"].xaxis.set_tick_params(rotation=0)
    start_time = pd.Timestamp(time[0])
    axs["A"].set_xlim(time[0], time[-1])
    axs["A"].yaxis.set_minor_locator(ticker.MultipleLocator(0.1))
    axs["A"].text(0.025, 0.88, "(a)", color="black", fontsize=15, transform = axs["A"].transAxes)
    axs["A"].legend(loc = 'best', frameon = 'True', fontsize = 10)

    #plt.savefig('time vs A_'+'.png', bbox_inches='tight')
    #files.download('time vs A_'+'.png')
    #plt.show()


    #plt.subplot(2,2,4)
    P = Pgas + (rotated[2] * 1e-9) ** 2 / (2 * 1.25663706212e-6) * 1e9

    a1 = aa[0:195]
    a2 = aa[195:363]
    P1 = P[0:195]
    P2 = P[195:363]
    #a1 = aa[0:127]
    #a2 = aa[127:317]
    #P1 = P[0:127]
    #P2 = P[127:317]

    #axs["PA"].scatter(a1, P1, color = 'red', s = 10)
    #axs["PA"].scatter(a2, P2, color = 'blue', s = 10)
    axs["PA"].plot(a1, P1, 'o-', color = 'red', label = '1st half path', markersize = 3.5, linewidth = 1.0)
    axs["PA"].plot(a2, P2, 'o-', color = 'blue', label = '2nd half path', markersize = 3.5, linewidth = 1.0)
    axs["PA"].set_xlabel(r"$|A/A_{peak}|$", fontsize=10)#, weight ='bold')
    axs["PA"].set_ylabel(r"$P_t$ [nPa]", fontsize=10)#, weight ='bold')
    axs["PA"].tick_params(axis = 'both', labelsize = 10)
    axs["PA"].xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
    axs["PA"].text(0.025, 0.88, "(b)", color="black", fontsize=15, transform = axs["PA"].transAxes)
    axs["PA"].legend(loc = 'best', frameon = 'True', fontsize = 10)

    #fig.savefig(str(y) + '_' + str(time[0])[0:4] + str(time[0])[5:7] + str(time[0])[8:10] + '_' + str(time[0])[11:13] + str(time[0])[14:16] + '_' + str(time[-1])[11:13] + str(time[-1])[14:16] + '_2.png')#.format(y=y))#, bbox_inches='tight')
    #files.download(str(y) + '_' + str(time[0])[0:4] + str(time[0])[5:7] + str(time[0])[8:10] + '_' + str(time[0])[11:13] + str(time[0])[14:16] + '_' + str(time[-1])[11:13] + str(time[-1])[14:16] + '_2.png')


